# Fokker-Planck Equation and Langevin Dynamics

## 1. Theory & Continuity Derivation
- Continuity equation: $\frac{\partial \rho}{\partial t} + \frac{\partial J}{\partial x} = 0$.
- Total current: $J = J_{\text{diff}} + J_{\text{drift}} = -D \frac{\partial \rho}{\partial x} - \rho \frac{dU}{dx}$.
- Fokker-Planck equation: $\frac{\partial \rho}{\partial t} = D \frac{\partial^2 \rho}{\partial x^2} + \frac{\partial}{\partial x}\left(\rho \frac{dU}{dx}\right)$.
- Equilibrium steady state: Gibbs-Boltzmann canonical distribution $\rho_{\text{eq}}(x) \propto e^{-U(x)/D}$.
- Microscopic Langevin equation: $m \frac{dv}{dt} = -\gamma v + \xi(t)$, with Einstein relation $D = k_B T / \gamma$.

## 2. EXACT CODE FROM QUIZ 2 REFERENCE PDF (Section 4.3)
1D Langevin Dynamics simulation integrating velocity and position under thermal white noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m = 1.0; gamma = 1.0; kB_T = 1.0; dt = 0.01; t_end = 50.0
nt = int(t_end / dt)
noise_std = np.sqrt(2 * gamma * kB_T * dt / (m**2))

v = np.zeros(nt); x = np.zeros(nt)
for i in range(nt - 1):
    v[i+1] = v[i] - (gamma / m) * v[i] * dt + noise_std * np.random.randn()
    x[i+1] = x[i] + v[i] * dt

t = np.arange(nt) * dt
plt.plot(t, x, label='Brownian Trajectory x(t)')
plt.xlabel('Time'); plt.ylabel('Position'); plt.title('1D Langevin Dynamics'); plt.grid(True); plt.show()


## 3. EXACT CODE FROM QUIZ 2 REFERENCE PDF (Section 4.4)
Fokker-Planck equation simulation with harmonic potential $U(x)$, drift + diffusion, and relaxation to potential minimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
dt = 0.01; dx = 0.05; D = 0.02; t_end = 100.0
N = 100; L = N * dx
t = np.arange(dt, t_end + dt, dt)
x = np.arange(1, N + 1) * dx

# Harmonic Potential U(x) = (k/2)*(x - L/2)^2
k_pot = 0.1
U = (k_pot / 2.0) * (x - L / 2.0)**2
Up = k_pot * (x - L / 2.0)
Up[0] = Up[-1] # Periodic boundary condition on force

# Initial concentration: displaced off-center Gaussian
rho = np.exp(-((x - L/4) / 1.0)**2)
rho_temp = np.zeros(N)

# Time Evolution
for step in range(len(t)):
    # Periodic boundary conditions
    rho[-1] = rho[1]
    rho[0] = rho[-2]
    
    rU = rho * Up
    for p in range(1, N - 1):
        diff_term = (D / (dx**2)) * (rho[p+1] + rho[p-1] - 2 * rho[p])
        drift_term = (rU[p+1] - rU[p-1]) / (2.0 * dx)
        rho_temp[p] = rho[p] + dt * (diff_term + drift_term)
        
    rho = rho_temp.copy()

# Plot relaxed steady state vs potential
plt.plot(x, rho / np.max(rho), 'b-', label='Normalized Concentration rho(x)')
plt.plot(x, U / np.max(U), 'r--', label='Normalized Potential U(x)')
plt.xlabel('x'); plt.ylabel('Profile'); plt.title('Fokker-Planck Relaxation to Potential Minimum')
plt.legend(); plt.grid(True); plt.show()


## 4. PROFESSOR'S IN-CLASS CODE (from `Notes/FokkerPlanck.ipynb`)
This is the code as directly written in `Notes/FokkerPlanck.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
dt = 0.01       # Temporal resolution
dx = 0.05       # Spatial resolution
D = 0.02        # Diffusion constant
t_end = 100     # Final time point

N = 100         # Number of grid points
L = N * dx      # System size

# Time and space vectors
t = np.arange(dt, t_end + dt, dt)
x = np.arange(1, N + 1) * dx

k = 0.1         # Sharpness parameter of the potential

# Initialize concentration fields
rho = np.zeros(N)
rho_0 = np.zeros(N)
rho_temp = np.zeros(N)

# Initial condition
rho_0 = np.exp(-((x - L / 4) / 1)**2)
rho = rho_0.copy()

# External potential and its derivative
U = (k / 2) * (x - L / 2)**2
Up = k * (x - L / 2)

# Periodic boundary condition
Up[0] = Up[-1]

# Store concentration at every time step
rho_T = np.zeros((N, len(t)))

# Time evolution
for i in range(len(t)):

    # Periodic boundary conditions
    rho[-1] = rho[1]
    rho[0] = rho[-2]

    # Product of concentration and potential derivative
    rU = rho * Up

    # Update concentration at interior grid points
    for p in range(1, N - 1):

        rho_temp[p] = rho[p] + dt * (D / dx**2 * (rho[p + 1] + rho[p - 1] - 2 * rho[p]) +
            (rU[p + 1] - rU[p - 1]) / (2 * dx)
        )
    # Update concentration
    rho = rho_temp.copy()

    # Store concentration profile
    rho_T[:, i] = rho

plt.plot(x[1:-1], rho_T[1:-1, 1000], label='Concentration')
plt.plot(x, 2 * U, label='2U (Potential)')
plt.legend()
plt.show()